In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes sentencepiece

In [ ]:
import json
import random

with open('diseases.json') as f:
    diseases = json.load(f)

TEMPLATES = [
    # 1
    "A {age}-year-old {gender} was seen in follow-up because of {critical}. "
    "The history suggested a chronic course with gradually progressive clinical concerns. "
    "Clinical assessment demonstrated {critical_2}. "
    "Additional findings included {supporting}. "
    "Associated features included {optional}. "
    "Supportive investigations included {labs}. "
    "The disease is associated with the {gene} gene and follows {inheritance} inheritance. "
    "Taken together, the findings formed a coherent syndrome pattern suitable for diagnostic training.",

    # 2
    "A {age}-year-old {gender} presented to the emergency department with {critical}. "
    "The onset was {onset_desc}. "
    "Examination revealed {critical_2} along with {supporting}. "
    "Laboratory and diagnostic workup included {labs}. "
    "The clinical picture was further complicated by {optional}. "
    "Genetic background: {gene} gene, {inheritance} inheritance pattern.",

    # 3
    "A {age}-year-old {gender} was referred for specialist evaluation of {critical}. "
    "Family members reported that the symptoms had become more noticeable over time. "
    "Examination showed {critical_2}. "
    "Additional findings included {supporting} and {optional}. "
    "Supportive investigations included {labs}. "
    "Known genetic association: {gene} ({inheritance}). "
    "The findings suggested a rare genetic syndrome requiring further workup.",

    # 4
    "A {age}-year-old {gender} child was brought to the clinic by parents concerned about {critical}. "
    "Developmental history was significant. "
    "Physical examination demonstrated {critical_2} and {supporting}. "
    "Associated features noted were {optional}. "
    "Investigations performed included {labs}. "
    "This condition typically presents at {onset_desc} onset and involves the {gene} gene.",

    # 5
    "A {age}-year-old {gender} adult presented with a long-standing history of {critical}. "
    "Symptoms had been present for several years before formal evaluation. "
    "On examination, {critical_2} was noted alongside {supporting}. "
    "Minor associated features included {optional}. "
    "Diagnostic investigations included {labs}. "
    "The condition follows {inheritance} inheritance and is linked to {gene}.",

    # 6
    "A {age}-year-old {gender} with a previous misdiagnosis was re-evaluated for {critical}. "
    "Prior treatments had been ineffective. "
    "Re-examination confirmed {critical_2} and revealed additional {supporting}. "
    "Incidental findings included {optional}. "
    "Workup included {labs}. "
    "Genetic testing identified involvement of {gene} with {inheritance} pattern. "
    "The revised clinical picture was consistent with a rare hereditary disorder.",

    # 7
    "A {age}-year-old {gender} infant was evaluated shortly after birth for {critical}. "
    "The neonatal course was complicated. "
    "Clinical findings included {critical_2} and {supporting}. "
    "Additional observations noted {optional}. "
    "Investigations included {labs}. "
    "Age of onset is typically {onset_desc}. Gene involved: {gene}.",

    # 8
    "A {age}-year-old {gender} was admitted to the ward for evaluation of {critical}. "
    "The admission was prompted by acute deterioration of a chronic condition. "
    "Inpatient assessment revealed {critical_2} and {supporting}. "
    "Supplementary findings were {optional}. "
    "Investigations carried out included {labs}. "
    "This rare disorder is caused by variants in {gene} and shows {inheritance} inheritance.",

    # 9
    "A {age}-year-old {gender} was referred to genetics following identification of {critical}. "
    "Family history was notable for similar features in a first-degree relative. "
    "Clinical evaluation confirmed {critical_2} and {supporting}. "
    "Additional syndromic features included {optional}. "
    "Investigations included {labs}. "
    "The {gene} gene is implicated, consistent with {inheritance} inheritance. "
    "Findings were consistent with a rare condition warranting genetic testing.",

    # 10
    "A {age}-year-old {gender} was discussed at a multidisciplinary team meeting presenting with {critical}. "
    "Multiple specialties had been involved in the care of this patient. "
    "Consensus examination findings included {critical_2} and {supporting}. "
    "Ancillary features noted were {optional}. "
    "Investigations reviewed included {labs}. "
    "Disease gene: {gene}. Inheritance: {inheritance}. Typical onset: {onset_desc}. "
    "The multidisciplinary assessment supported a rare complex syndrome diagnosis.",

    # 11
    "A {age}-year-old {gender} was referred to a specialist clinic after developing {critical}. "
    "The symptoms had gradually increased in severity over time. "
    "Physical examination revealed {critical_2}, with additional evidence of {supporting}. "
    "Other associated manifestations included {optional}. "
    "Relevant investigations showed {labs}. "
    "Molecular evaluation implicated the {gene} gene, which is associated with {inheritance} inheritance.",

    # 12
    "A {age}-year-old {gender} presented with concerns regarding {critical}. "
    "The clinical history indicated an onset described as {onset_desc}. "
    "Detailed examination demonstrated {critical_2}. "
    "Further assessment identified {supporting} and {optional}. "
    "Diagnostic studies included {labs}. "
    "The overall presentation was compatible with a disorder involving {gene} and showing {inheritance} inheritance.",

    # 13
    "A {age}-year-old {gender} underwent evaluation after persistent {critical} raised concern for an underlying genetic disorder. "
    "Symptoms had followed a {onset_desc} course. "
    "Clinical examination demonstrated {critical_2}. "
    "Additional physical findings included {supporting}. "
    "Associated manifestations were described as {optional}. "
    "Investigations revealed {labs}. "
    "The suspected condition has been linked to pathogenic variants in {gene} and follows {inheritance} inheritance.",

    # 14
    "A {age}-year-old {gender} was evaluated because of recurrent episodes of {critical}. "
    "The clinical history was notable for {onset_desc} onset. "
    "During assessment, {critical_2} was identified. "
    "Other relevant findings included {supporting}. "
    "The patient also demonstrated {optional}. "
    "Laboratory and imaging investigations included {labs}. "
    "The phenotype was associated with the {gene} gene and an {inheritance} inheritance pattern.",

    # 15
    "A {age}-year-old {gender} presented for assessment of unexplained {critical}. "
    "The condition had first become apparent with {onset_desc} onset. "
    "Examination identified {critical_2} together with {supporting}. "
    "Additional clinical features included {optional}. "
    "The diagnostic evaluation consisted of {labs}. "
    "The findings raised suspicion for a hereditary condition involving {gene} with {inheritance} inheritance.",

    # 16
    "A {age}-year-old {gender} was evaluated in a specialist setting for {critical}. "
    "According to the clinical history, symptoms had developed with {onset_desc}. "
    "Physical examination showed {critical_2}. "
    "Further examination revealed {supporting}, while other associated features included {optional}. "
    "The workup included {labs}. "
    "The disorder is genetically associated with {gene} and is inherited in an {inheritance} pattern.",

    # 17
    "A {age}-year-old {gender} was referred after clinicians noted {critical}. "
    "The presentation was considered unusual because of its {onset_desc} onset. "
    "On detailed examination, {critical_2} was present. "
    "Additional clinical manifestations included {supporting}. "
    "The patient also had {optional}. "
    "Investigations performed during the evaluation included {labs}. "
    "Genetic findings implicated {gene}, consistent with {inheritance} inheritance.",

    # 18
    "A {age}-year-old {gender} underwent a comprehensive clinical assessment for {critical}. "
    "The disorder had manifested at {onset_desc}. "
    "Clinical examination demonstrated {critical_2} with accompanying {supporting}. "
    "Several additional features, including {optional}, were documented. "
    "Relevant investigations included {labs}. "
    "The clinical phenotype was consistent with a disorder associated with {gene} and {inheritance} inheritance.",

    # 19
    "A {age}-year-old {gender} was brought for further investigation of persistent {critical}. "
    "Review of the history suggested {onset_desc} onset and a progressive clinical course. "
    "Examination revealed {critical_2}. "
    "Additional findings were notable for {supporting}. "
    "Other associated manifestations included {optional}. "
    "Diagnostic workup included {labs}. "
    "The condition has been associated with mutations in {gene} and follows {inheritance} inheritance.",

    # 20
    "A {age}-year-old {gender} was assessed following recognition of {critical}. "
    "The symptoms were reported to have appeared with {onset_desc} onset. "
    "Clinical examination demonstrated {critical_2} and evidence of {supporting}. "
    "Additional features included {optional}. "
    "Investigations performed as part of the evaluation included {labs}. "
    "The suspected syndrome is associated with the {gene} gene and follows an {inheritance} inheritance pattern.",

    # 21
    "A {age}-year-old {gender} presented for evaluation of a suspected inherited disorder characterized by {critical}. "
    "The history was consistent with {onset_desc} onset. "
    "Examination demonstrated {critical_2}. "
    "Other clinically relevant findings included {supporting}. "
    "Associated manifestations were also noted, including {optional}. "
    "Investigations included {labs}. "
    "The underlying genetic association involves {gene} with {inheritance} inheritance.",

    # 22
    "A {age}-year-old {gender} was evaluated after developing a combination of {critical}. "
    "The clinical course began with {onset_desc} onset. "
    "Assessment showed {critical_2}, accompanied by {supporting}. "
    "Additional associated findings included {optional}. "
    "The diagnostic workup incorporated {labs}. "
    "The resulting phenotype was suggestive of a rare disorder involving {gene} and following {inheritance} inheritance.",

    # 23
    "A {age}-year-old {gender} was referred for evaluation of {critical} in the setting of a possible syndromic disorder. "
    "The symptoms were first recognized at {onset_desc}. "
    "Physical assessment identified {critical_2}. "
    "Further examination revealed {supporting} and other features such as {optional}. "
    "Laboratory and diagnostic investigations included {labs}. "
    "The suspected disorder is associated with {gene} and demonstrates {inheritance} inheritance.",

    # 24
    "A {age}-year-old {gender} presented with a constellation of findings centered around {critical}. "
    "The clinical history indicated {onset_desc} onset. "
    "On examination, {critical_2} was observed. "
    "Additional abnormalities included {supporting}. "
    "Other associated findings were described as {optional}. "
    "Investigations performed included {labs}. "
    "These findings were compatible with a rare genetic condition involving {gene} and {inheritance} inheritance.",

    # 25
    "A {age}-year-old {gender} was referred for further assessment after the development of {critical}. "
    "The condition was reported to have an {onset_desc} onset. "
    "Clinical evaluation identified {critical_2} and additional findings of {supporting}. "
    "Associated clinical features included {optional}. "
    "The investigation panel included {labs}. "
    "The overall phenotype suggested a genetic syndrome related to {gene} with {inheritance} inheritance.",

    # 26
    "A {age}-year-old {gender} was evaluated because of a persistent clinical concern involving {critical}. "
    "Historical information suggested {onset_desc} onset. "
    "Examination confirmed {critical_2} and demonstrated {supporting}. "
    "Additional manifestations included {optional}. "
    "Relevant diagnostic studies showed {labs}. "
    "Genetic evaluation focused on {gene}, which is associated with {inheritance} inheritance.",

    # 27
    "A {age}-year-old {gender} presented to a tertiary care center with {critical}. "
    "The clinical history indicated that symptoms began with {onset_desc}. "
    "Specialist examination revealed {critical_2}. "
    "Additional examination findings included {supporting}. "
    "The patient also exhibited {optional}. "
    "Diagnostic investigations included {labs}. "
    "The condition is genetically linked to {gene} and follows an {inheritance} inheritance pattern.",

    # 28
    "A {age}-year-old {gender} was referred for diagnostic clarification of {critical}. "
    "The clinical presentation had an {onset_desc} onset. "
    "Examination demonstrated {critical_2}, with further findings of {supporting}. "
    "Other associated manifestations included {optional}. "
    "Investigations undertaken during evaluation included {labs}. "
    "The suspected diagnosis was a rare hereditary disorder associated with {gene} and {inheritance} inheritance.",

    # 29
    "A {age}-year-old {gender} underwent genetic evaluation after presenting with {critical}. "
    "The history suggested that the condition had begun at {onset_desc}. "
    "Clinical examination revealed {critical_2} and {supporting}. "
    "Additional associated features included {optional}. "
    "Laboratory and diagnostic testing included {labs}. "
    "The clinical findings were considered consistent with a disorder involving the {gene} gene and {inheritance} inheritance.",

    # 30
    "A {age}-year-old {gender} was evaluated for a complex clinical presentation dominated by {critical}. "
    "The history indicated {onset_desc} onset with subsequent evolution of additional manifestations. "
    "Examination demonstrated {critical_2}. "
    "Further findings included {supporting} and {optional}. "
    "The diagnostic evaluation included {labs}. "
    "The overall presentation supported consideration of a rare genetic syndrome associated with {gene} and {inheritance} inheritance."
]

GENDERS       = ['male', 'female', 'boy', 'girl', 'man', 'woman']
ONSET_FALLBACK = ['neonatal', 'childhood', 'adolescent', 'adult']

def format_symptoms_for_prompt(critical, supporting, optional):
    """
    Format symptoms with their frequency meaning clearly stated.
    BioGPT learns: very frequent = must appear, frequent = likely, occasional = rare
    """
    lines = []
    lines.append('Symptoms (very frequent - almost always present, score 0.90):')
    lines.append('  ' + ', '.join(critical)   if critical   else '  none')

    lines.append('Symptoms (frequent - often present, score 0.55):')
    lines.append('  ' + ', '.join(supporting) if supporting else '  none')

    lines.append('Symptoms (occasional - sometimes present, score 0.17):')
    lines.append('  ' + ', '.join(optional)   if optional   else '  none')

    return '\n'.join(lines)

def disease_to_training_samples(disease):
    name        = disease['disease_name']
    symptoms    = disease.get('symptoms', [])
    labs        = disease.get('lab_findings', []) or ['clinical examination']
    genes       = disease.get('gene_involved', []) or ['unknown']
    inheritance = disease.get('inheritance', '')   or 'unknown'
    age_onset   = disease.get('age_of_onset', [])  or ONSET_FALLBACK

    # Sort by frequency score descending
    scored = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    # Split into 3 frequency tiers
    critical   = [s['name'] for s in scored if s['frequency_score'] >= 0.9]
    supporting = [s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.9]
    optional   = [s['name'] for s in scored if s['frequency_score'] < 0.55]

    # Fallbacks
    if not critical:   critical   = [s['name'] for s in scored[:3]]
    if not supporting: supporting = critical[:2]
    if not optional:   optional   = supporting[:1]
    if not critical:   critical   = ['unspecified findings']

    samples = []
    for idx, template in enumerate(TEMPLATES):

        # Sample subsets for variety across templates
        crit_sample  = random.sample(critical,   min(3, len(critical)))
        supp_sample  = random.sample(supporting, min(3, len(supporting)))
        opt_sample   = random.sample(optional,   min(2, len(optional)))
        lab_sample   = random.sample(labs,       min(2, len(labs)))
        gene         = random.choice(genes)
        onset        = random.choice(age_onset)

        # Age matched to onset
        onset_str = str(onset).lower()
        if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
            age = random.choice([0, 1, 2])
        elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
            age = random.choice([3, 5, 7, 8, 10, 12])
        elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
            age = random.choice([13, 15, 16, 17])
        elif any(w in onset_str for w in ['adult', 'elder', 'old']):
            age = random.choice([25, 30, 35, 42, 50, 60])
        else:
            age = random.choice([5, 8, 12, 15, 25, 35])

        gender = random.choice(GENDERS)

        scenario = template.format(
            age         = age,
            gender      = gender,
            critical    = ', '.join(crit_sample[:2]),
            critical_2  = ', '.join(crit_sample),
            supporting  = ', '.join(supp_sample),
            optional    = ', '.join(opt_sample),
            labs        = ' and '.join(lab_sample),
            onset_desc  = onset,
            gene        = gene,
            inheritance = inheritance,
        )

        # ── Prompt now includes full frequency-labelled symptom list ──
        symptom_block = format_symptoms_for_prompt(critical, supporting, optional)

        prompt = (
            f"Disease: {name}\n"
            f"Gene: {gene}\n"
            f"Inheritance: {inheritance}\n"
            f"Onset: {onset}\n"
            f"Lab findings: {', '.join(labs)}\n"
            f"{symptom_block}\n"
            f"Generate a clinical case scenario:\n\n"
        )

        samples.append({
            'disease_name'   : name,
            'gene'           : gene,
            'inheritance'    : inheritance,
            'onset'          : str(onset),
            'prompt'         : prompt,
            'completion'     : scenario,
            'full_text'      : prompt + scenario,
            'template_idx'   : idx + 1,
        })

    return samples

# ── Generate all ──────────────────────────────────────────────────
print('Converting 483 diseases × 10 templates...')
all_samples = []
for d in diseases:
    all_samples.extend(disease_to_training_samples(d))

print(f'Total training samples: {len(all_samples)}')
print()

# Show one complete sample so you can verify it looks correct
s = all_samples[5]
print('=' * 60)
print('FULL PROMPT BioGPT WILL SEE:')
print('=' * 60)
print(s['prompt'])
print('=' * 60)
print('COMPLETION BioGPT WILL GENERATE:')
print('=' * 60)
print(s['completion'])

with open('biogpt_training_data.json', 'w') as f:
    json.dump(all_samples, f, indent=2)
print('\nSaved: biogpt_training_data.json')

from google.colab import files
files.download('biogpt_training_data.json')

In [ ]:
import torch
print(torch.cuda.is_available())        
print(torch.cuda.get_device_name(0))    

In [ ]:
import torch
print(torch.cuda.is_available())        
print(torch.cuda.get_device_name(0))    

In [ ]:
!pip install -q sacremoses

In [ ]:
!pip install -U torchao

In [ ]:
import torch
import json
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

# ── Load training data ────────────────────────────────────────────
with open('biogpt_training_data.json') as f:
    raw = json.load(f)

texts = [r['full_text'] for r in raw]
print(f'Total training samples : {len(texts)}')

# ── Load BioGPT ───────────────────────────────────────────────────
print('Loading BioGPT tokenizer and model...')
MODEL_NAME = 'microsoft/biogpt'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.float16,
    device_map  = 'auto',
)
print('BioGPT loaded.')

# ── Apply LoRA ────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 8,
    lora_alpha     = 16,
    lora_dropout   = 0.1,
    target_modules = ['q_proj', 'v_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Tokenise ──────────────────────────────────────────────────────
def tokenise(examples):
    tokens = tokenizer(
        examples['text'],
        truncation = True,
        max_length = 512,
        padding    = 'max_length',
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

dataset = Dataset.from_dict({'text': texts})
dataset = dataset.map(tokenise, batched=True, remove_columns=['text'])
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f'Train : {len(dataset["train"])}')
print(f'Eval  : {len(dataset["test"])}')

# ── Training arguments ────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = './biogpt_lora',
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,
    warmup_steps                = 100,
    learning_rate               = 2e-4,
    fp16                        = True,
    logging_steps               = 50,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
)

# ── Train ─────────────────────────────────────────────────────────
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = dataset['train'],
    eval_dataset  = dataset['test'],
    data_collator = DataCollatorForLanguageModeling(
        tokenizer = tokenizer,
        mlm       = False,
    ),
)

print('Starting fine-tuning — this will take 2 to 4 hours...')
print('Do not close this tab.')
trainer.train()
print('Fine-tuning complete!')

# ── Save ──────────────────────────────────────────────────────────
model.save_pretrained('./biogpt_lora_final')
tokenizer.save_pretrained('./biogpt_lora_final')
print('Model saved to ./biogpt_lora_final')

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)
print(f'Diseases loaded: {len(diseases)}')

# ── Build prompt from disease record ─────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
def generate_scenario(prompt_text, max_new_tokens=250):
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()
    return result

# ── Generate 10 scenarios per disease ────────────────────────────
all_scenarios = []
print(f'Starting generation: {len(diseases)} diseases × 10 variants = {len(diseases)*10} total')
print()

for i, disease in enumerate(diseases):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, max_new_tokens=250)
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    # Notify every 100 diseases
    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/{len(diseases)}] diseases done — {len(all_scenarios)} scenarios generated')

print(f'\nAll done! Total scenarios: {len(all_scenarios)}')

# ── Save to file ──────────────────────────────────────────────────
with open('biogpt_generated_scenarios.json', 'w') as f:
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_generated_scenarios.json')

# ── Download ──────────────────────────────────────────────────────
from google.colab import files
files.download('biogpt_generated_scenarios.json')

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)

batch = diseases[0:100]    # ← first 100 only
print(f'Diseases loaded : {len(diseases)} total')
print(f'This run        : {len(batch)} diseases × 10 = {len(batch)*10} scenarios')
print()

# ── Build prompt ──────────────────────────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
import re
import random

GENDERS = ['boy', 'girl', 'man', 'woman', 'male', 'female']

def generate_scenario(prompt_text, disease, max_new_tokens=250):
    # Pick age based on onset
    onset_str = str((disease.get('age_of_onset') or [''])[0]).lower()
    if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
        age = random.randint(0, 2)
    elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
        age = random.randint(3, 12)
    elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
        age = random.randint(13, 17)
    elif any(w in onset_str for w in ['adult', 'elder']):
        age = random.randint(25, 65)
    else:
        age = random.randint(5, 50)

    gender = random.choice(GENDERS)
    forced_start = f'A {age}-year-old {gender}'

    # Let BioGPT generate freely
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()

    # ── Remove ANY broken age pattern at the start ────────────────
    # Catches: "r-old", "ar-old", "3r-old", "-year-old", "9-year-old boy ar-old"
    result = re.sub(r'^[\w\d]*-?r?-?year-old\s+\w+\s+', '', result)
    result = re.sub(r'^[\w\d]*-?r?-?old\s+', '', result)
    result = result.strip()

    # ── Prepend our clean forced start ────────────────────────────
    result = forced_start + ' ' + result

    return result

# ── Generate ──────────────────────────────────────────────────────
all_scenarios = []

for i, disease in enumerate(batch):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, disease)  # ← pass disease here
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/100] done — {len(all_scenarios)} scenarios')

print(f'\nDone! Total scenarios: {len(all_scenarios)}')

# ── Save ──────────────────────────────────────────────────────────
with open('biogpt_.1_diseases_0_to_99.json', 'w') as f:
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_day1_diseases_0_to_99.json')

from google.colab import files
files.download('biogpt_day1_diseases_0_to_99.json')

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)

batch = diseases[100:200]
print(f'Diseases loaded : {len(diseases)} total')
print(f'This run        : {len(batch)} diseases × 10 = {len(batch)*10} scenarios')
print()

# ── Build prompt ──────────────────────────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
import re
import random

GENDERS = ['boy', 'girl', 'man', 'woman', 'male', 'female']

def generate_scenario(prompt_text, disease, max_new_tokens=250):
    # Pick age based on onset
    onset_str = str((disease.get('age_of_onset') or [''])[0]).lower()
    if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
        age = random.randint(0, 2)
    elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
        age = random.randint(3, 12)
    elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
        age = random.randint(13, 17)
    elif any(w in onset_str for w in ['adult', 'elder']):
        age = random.randint(25, 65)
    else:
        age = random.randint(5, 50)

    gender = random.choice(GENDERS)
    forced_start = f'A {age}-year-old {gender}'

    # Let BioGPT generate freely
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()

    # ── Remove ANY broken age pattern at the start ────────────────
    # Catches: "r-old", "ar-old", "3r-old", "-year-old", "9-year-old boy ar-old"
    result = re.sub(r'^[\w\d]*-?r?-?year-old\s+\w+\s+', '', result)
    result = re.sub(r'^[\w\d]*-?r?-?old\s+', '', result)
    result = result.strip()

    # ── Prepend our clean forced start ────────────────────────────
    result = forced_start + ' ' + result

    return result

# ── Generate ──────────────────────────────────────────────────────
all_scenarios = []

for i, disease in enumerate(batch):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, disease)  # ← pass disease here
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/100] done — {len(all_scenarios)} scenarios')

print(f'\nDone! Total scenarios: {len(all_scenarios)}')

# ── Save ──────────────────────────────────────────────────────────
with open('biogpt_diseases_100_to_199.json', 'w') as f:   # ← 
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_diseases_100_to_199.json')

from google.colab import files
files.download('biogpt_diseases_100_to_199.json')   # ← 

#200-300

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)

batch = diseases[200:300]
print(f'Diseases loaded : {len(diseases)} total')
print(f'This run        : {len(batch)} diseases × 10 = {len(batch)*10} scenarios')
print()

# ── Build prompt ──────────────────────────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
import re
import random

GENDERS = ['boy', 'girl', 'man', 'woman', 'male', 'female']

def generate_scenario(prompt_text, disease, max_new_tokens=250):
    # Pick age based on onset
    onset_str = str((disease.get('age_of_onset') or [''])[0]).lower()
    if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
        age = random.randint(0, 2)
    elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
        age = random.randint(3, 12)
    elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
        age = random.randint(13, 17)
    elif any(w in onset_str for w in ['adult', 'elder']):
        age = random.randint(25, 65)
    else:
        age = random.randint(5, 50)

    gender = random.choice(GENDERS)
    forced_start = f'A {age}-year-old {gender}'

    # Let BioGPT generate freely
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()

    # ── Remove ANY broken age pattern at the start ────────────────
    # Catches: "r-old", "ar-old", "3r-old", "-year-old", "9-year-old boy ar-old"
    result = re.sub(r'^[\w\d]*-?r?-?year-old\s+\w+\s+', '', result)
    result = re.sub(r'^[\w\d]*-?r?-?old\s+', '', result)
    result = result.strip()

    # ── Prepend our clean forced start ────────────────────────────
    result = forced_start + ' ' + result

    return result

# ── Generate ──────────────────────────────────────────────────────
all_scenarios = []

for i, disease in enumerate(batch):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, disease)  # ← pass disease here
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/100] done — {len(all_scenarios)} scenarios')

print(f'\nDone! Total scenarios: {len(all_scenarios)}')

# ── Save ──────────────────────────────────────────────────────────
with open('biogpt_diseases_200_to_299.json', 'w') as f:   # ← 
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_diseases_200_to_299.json')

from google.colab import files
files.download('biogpt_diseases_200_to_299.json')   # ← 

#300-400

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)

batch = diseases[300:400]
print(f'Diseases loaded : {len(diseases)} total')
print(f'This run        : {len(batch)} diseases × 10 = {len(batch)*10} scenarios')
print()

# ── Build prompt ──────────────────────────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
import re
import random

GENDERS = ['boy', 'girl', 'man', 'woman', 'male', 'female']

def generate_scenario(prompt_text, disease, max_new_tokens=250):
    # Pick age based on onset
    onset_str = str((disease.get('age_of_onset') or [''])[0]).lower()
    if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
        age = random.randint(0, 2)
    elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
        age = random.randint(3, 12)
    elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
        age = random.randint(13, 17)
    elif any(w in onset_str for w in ['adult', 'elder']):
        age = random.randint(25, 65)
    else:
        age = random.randint(5, 50)

    gender = random.choice(GENDERS)
    forced_start = f'A {age}-year-old {gender}'

    # Let BioGPT generate freely
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()

    # ── Remove ANY broken age pattern at the start ────────────────
    # Catches: "r-old", "ar-old", "3r-old", "-year-old", "9-year-old boy ar-old"
    result = re.sub(r'^[\w\d]*-?r?-?year-old\s+\w+\s+', '', result)
    result = re.sub(r'^[\w\d]*-?r?-?old\s+', '', result)
    result = result.strip()

    # ── Prepend our clean forced start ────────────────────────────
    result = forced_start + ' ' + result

    return result

# ── Generate ──────────────────────────────────────────────────────
all_scenarios = []

for i, disease in enumerate(batch):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, disease)  # ← pass disease here
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/100] done — {len(all_scenarios)} scenarios')

print(f'\nDone! Total scenarios: {len(all_scenarios)}')

# ── Save ──────────────────────────────────────────────────────────
with open('biogpt_diseases_300_to_399.json', 'w') as f:   # ← 
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_diseases_300_to_399.json')

from google.colab import files
files.download('biogpt_diseases_300_to_399.json')   # ← 

400-

In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Load fine-tuned model ─────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained('./biogpt_lora_final')
base_model = AutoModelForCausalLM.from_pretrained(
    'microsoft/biogpt',
    dtype      = torch.float16,
    device_map = 'auto'
)
model = PeftModel.from_pretrained(base_model, './biogpt_lora_final')
model.eval()
print('Model loaded.')

# ── Load diseases ─────────────────────────────────────────────────
with open('diseases.json') as f:
    diseases = json.load(f)

batch = diseases[400:]
print(f'Diseases loaded : {len(diseases)} total')
print(f'This run        : {len(batch)} diseases × 10 = {len(batch)*10} scenarios')
print()

# ── Build prompt ──────────────────────────────────────────────────
def build_prompt(disease):
    name        = disease['disease_name']
    gene        = (disease.get('gene_involved') or ['unknown'])[0]
    inheritance = disease.get('inheritance') or 'unknown'
    onset       = (disease.get('age_of_onset') or ['unknown'])[0]
    labs        = ', '.join(disease.get('lab_findings') or ['clinical examination'])

    symptoms = disease.get('symptoms', [])
    scored   = [s for s in symptoms if s.get('frequency_score') is not None]
    scored.sort(key=lambda x: x['frequency_score'], reverse=True)

    critical   = ', '.join(s['name'] for s in scored if s['frequency_score'] >= 0.90)[:200]
    supporting = ', '.join(s['name'] for s in scored if 0.55 <= s['frequency_score'] < 0.90)[:200]
    optional   = ', '.join(s['name'] for s in scored if s['frequency_score'] < 0.55)[:200]

    return f"""Disease: {name}
Gene: {gene}
Inheritance: {inheritance}
Onset: {onset}
Lab findings: {labs}
Symptoms (very frequent - almost always present, score 0.90):
  {critical or 'unspecified'}
Symptoms (frequent - often present, score 0.55):
  {supporting or 'unspecified'}
Symptoms (occasional - sometimes present, score 0.17):
  {optional or 'unspecified'}
Generate a clinical case scenario:

"""

# ──  ─────────────────────────────────────────
import re
import random

GENDERS = ['boy', 'girl', 'man', 'woman', 'male', 'female']

def generate_scenario(prompt_text, disease, max_new_tokens=250):
    # Pick age based on onset
    onset_str = str((disease.get('age_of_onset') or [''])[0]).lower()
    if any(w in onset_str for w in ['neonat', 'birth', 'infant']):
        age = random.randint(0, 2)
    elif any(w in onset_str for w in ['child', 'pediatric', 'school']):
        age = random.randint(3, 12)
    elif any(w in onset_str for w in ['adoles', 'teen', 'juvenile']):
        age = random.randint(13, 17)
    elif any(w in onset_str for w in ['adult', 'elder']):
        age = random.randint(25, 65)
    else:
        age = random.randint(5, 50)

    gender = random.choice(GENDERS)
    forced_start = f'A {age}-year-old {gender}'

    # Let BioGPT generate freely
    inputs = tokenizer(
        prompt_text,
        return_tensors = 'pt'
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )

    full   = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full[len(prompt_text):].strip()

    # ── Remove ANY broken age pattern at the start ────────────────
    # Catches: "r-old", "ar-old", "3r-old", "-year-old", "9-year-old boy ar-old"
    result = re.sub(r'^[\w\d]*-?r?-?year-old\s+\w+\s+', '', result)
    result = re.sub(r'^[\w\d]*-?r?-?old\s+', '', result)
    result = result.strip()

    # ── Prepend our clean forced start ────────────────────────────
    result = forced_start + ' ' + result

    return result

# ── Generate ──────────────────────────────────────────────────────
all_scenarios = []

for i, disease in enumerate(batch):
    prompt = build_prompt(disease)

    for variant in range(1, 11):
        scenario_text = generate_scenario(prompt, disease)  # ← pass disease here
        all_scenarios.append({
            'scenario_id'       : f'BGPT-{i+1:04d}-{variant:02d}',
            'disease_name'      : disease['disease_name'],
            'orpha_code'        : disease.get('orpha_code', ''),
            'omim_id'           : disease.get('omim_id', ''),
            'scenario_text'     : scenario_text,
            'variant_index'     : variant,
            'generation_method' : 'biogpt_lora_finetuned',
        })

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/100] done — {len(all_scenarios)} scenarios')

print(f'\nDone! Total scenarios: {len(all_scenarios)}')

# ── Save ──────────────────────────────────────────────────────────
with open('biogpt_diseases_400_to_483.json', 'w') as f:   # ← 
    json.dump(all_scenarios, f, indent=2)
print('Saved: biogpt_diseases_400_to_483.json')

from google.colab import files
files.download('biogpt_diseases_400_to_483.json')   # ← 